In [ ]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

if not catalog:
    raise ValueError(
        "Il parametro 'catalog' non è stato valorizzato."
    )

if not schema:
    raise ValueError(
        "Il parametro 'schema' non è stato valorizzato."
    )

silver_table = f"{catalog}.{schema}.silver_breweries"
gold_table = f"{catalog}.{schema}.gold_breweries"
agg_table = f"{catalog}.{schema}.agg_breweries"

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Silver: {silver_table}")
print(f"Gold: {gold_table}")
print(f"Agg: {agg_table}")

In [ ]:
def assert_table_exist(table_name: str) ->str:
    if not spark.catalog.tableExists(table_name):
        raise AssertionError(
            f"La tabella {table_name} non esiste"
        )

    print(f"OK - La tabella {table_name} esiste.")

assert_table_exists(silver_table)
assert_table_exists(gold_table)

In [ ]:
def assert_table_not_empty(table_name: str) ->str:
    row_count = spark.table(table_name).count()

    if row_count == 0:
        raise AssertionError (
            f"La tabella {table_name} è vuota."
        )

    print(
        f"OK - La tabella {table_name} contiene"
        f"{row_count} record"
    )

    return row_count

silver_count = assert_table_not_empty(silver_table)
gold_count = assert_table_not_empty(gold_table)

In [ ]:
silver_df = spark.table(silver_table)

null_brewery_sk_count = (
    silver_df
        .filter(F.col("brewery_sk").isNull())
        .count()
)

if null_brewery_sk_count > 0:
    raise AssertionError(
        "La tabella Silver contiene "
        f"{null_brewery_sk_count} brewery_sk nulle."
    )

print("OK - Nessuna brewery_sk nulla nella Silver.")

In [ ]:
required_silver_columns = {
    "brewery_sk",
    "id",
    "name",
    "country",
    "ingestion_ts",
}

"""
required_silver_columns = set([
    "brewery_sk",
    "id",
    "name",
    "country",
    "ingestion_ts",
])
"""

missing_columns = {
    required_silver_columns - set(silver_df.columns)
}

if missing_columns:
    raise AssertionError(
        "Nella Silver mancano le colonne: "
        f"{sorted(missing_columns)}"
    )

print("OK - La Silver contiene tutte le colonne richieste.")

In [ ]:
duplicate_versions_count = (
    silver_df
    .groupBy("id", "ingestion_ts")
    .count()
    .agg(F.sum(F.when(F.col("count") > 1), 1 ).otherwise(0) ) 
    .collect[0][0]
)

if duplicate_versions_count > 0:
    raise AssertionError(
        "La Silver contiene "
        f"{duplicate_versions_count} combinazioni duplicate "
        "di id e ingestion_ts."
    )

print(
    "OK - Nessun duplicato tecnico "
    "sulla coppia id + ingestion_ts."
)

In [ ]:
gold_df = spark.table(gold_table)

required_gold_columns = {
    "id",
    "__START_AT",
    "__END_AT",
}

missing_gold_columns = (
    required_gold_columns - set(gold_df.columnns)
)

if missing_gold_columns:
    raise AssertionError(
        "Nella Gold mancano le colonne SCD2: "
        f"{sorted(missing_gold_columns)}"
    )

print("OK - La Gold contiene le colonne SCD Type 2.")

In [ ]:
duplicate_current_versions = (
    gold_df
    .where(F.col("__END_AT").isNull())
    .groupBy("id")
    .agg(F.sum(F.when(F.col("count")> 1, 1) ).otherwise(0) )
    .collect()[0][0]
)

if duplicate_current_versions > 0:
    raise AssertionError(
        "Sono presenti "
        f"{duplicate_current_versions} brewery con più "
        "di una versione corrente nella Gold."
    )

print(
    "OK - Ogni brewery ha al massimo "
    "una versione corrente."
)

In [0]:
tables = [
    "bronze_breweries",
    "bronze_breweries_job_config",
    "cdc_breweries_events",
    "silver_staging_breweries",
    "silver_breweries",
    "gold_breweries",
    "agg_breweries",
]

for table in tables:
    full_name = f"{base}.{table}"

    try:
        count_rows = spark.table(full_name).count()
        print(f"Table {full_name} exists with {count_rows} rows.")
    except Exception as e:
        print(f"Table {full_name} does not exist or cannot be accessed. Error: {e}")

***
**Verificare current_page**

In [0]:
display(
    spark.table(f"{base}.bronze_breweries_job_config")
)

***
**Verificare lo storico SCD2**

In [0]:
storico = (
    spark.table(f"{base}.gold_breweries")
    .select(
        "id",
        "name",
        "phone",
        "street",
        "__START_AT",
        "__END_AT",
    )
    .orderBy("id", "__START_AT")
)

display(storico)

***
**Breweries Version**

In [0]:

gold_df = spark.table(gold_table)

display(
    gold_df
    .groupBy("id")
    .agg(
        F.count("*").alias("number_versions")
    )
    .where(F.col("number_versions") > 1)
    .orderBy(F.col("number_versions").desc())
)

In [0]:
display(
    gold_df
    .where(F.col("__END_AT").isNull())
)

In [0]:
invalid_current_versions = (
    gold_df
    .where(F.col("__END_AT").isNull())
    .groupBy("id")
    .count()
    .where(F.col("count") != 1)
)

invalid_current_versions_count = invalid_current_versions.count()

if invalid_current_versions_count > 0:
    display(invalid_current_versions)

    raise AssertionError(
        "Sono presenti "
        f"{invalid_current_versions_count} brewery con un numero "
        "di versioni correnti diverso da 1."
    )

print("OK - Ogni brewery ha esattamente una versione corrente.")

***
**Verify agg_breweries**

In [0]:
display(
    spark.table(f"{base}.agg_breweries")
    .orderBy(F.col("num_breweries").desc())
)

In [ ]:
print("=" * 60)
print("INTEGRATION TEST COMPLETATO CON SUCCESSO")
print(f"Silver records: {silver_count}")
print(f"Gold records: {gold_count}")
print("=" * 60)